# Analyses descriptives extraday — Futures index SXE / DAX

Ce notebook complète l'analyse intraday. Il travaille au niveau **jour / session** : stabilité jour-par-jour, overnight, gaps, jours extrêmes, roll, jours macro, persistance inter-journalière et comparaison DAX/SXE.

Données attendues :

- `timestamp` : horodatage du trade ;
- `price` : prix du trade ;
- `qté` : quantité exécutée ;
- `sens` : direction agressive, `+1` achat agressif, `-1` vente agressive ;
- `mid` : mid-price au moment du trade ;
- `bid` : best bid ;
- `ask` : best ask.

Objectif : préparer une base solide avant Hasbrouck VAR, Hawkes, point processes ou modèles prédictifs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1. Paramètres utilisateur

Adapte les paramètres au contrat.

La tick size permet de convertir les mouvements de prix en ticks :

$$
r_t^{ticks} = \frac{mid_t - mid_{t-1}}{tick\_size}
$$


In [ ]:
TIMESTAMP_COL = "timestamp"
PRICE_COL = "price"
VOLUME_COL = "qté"
SIDE_COL = "sens"
MID_COL = "mid"
BID_COL = "bid"
ASK_COL = "ask"

tick_size = 0.5       # à adapter au contrat
multiplier = 25.0     # valeur du point, à adapter
timezone = None       # ex: "Europe/Paris" si nécessaire
INTRADAY_BUCKET = "1min"


## 2. Chargement des données

Décommente et adapte la cellule suivante à ton fichier.


In [ ]:
# Exemple :
# df = pd.read_csv("trades_dax.csv")
# df = pd.read_parquet("trades_dax.parquet")

# df.head()


## 3. Préparation des données

Cette fonction nettoie les données et crée les variables de base : spread, rendement en ticks, signed volume, notional, date de session.


In [ ]:
def prepare_trade_df(
    df,
    timestamp_col=TIMESTAMP_COL,
    price_col=PRICE_COL,
    volume_col=VOLUME_COL,
    side_col=SIDE_COL,
    mid_col=MID_COL,
    bid_col=BID_COL,
    ask_col=ASK_COL,
    tick_size=tick_size,
    multiplier=multiplier,
    timezone=timezone
):
    out = df.copy()

    if not np.issubdtype(out[timestamp_col].dtype, np.datetime64):
        if np.issubdtype(out[timestamp_col].dtype, np.number):
            med = out[timestamp_col].dropna().median()
            if med > 1e12:
                out[timestamp_col] = pd.to_datetime(out[timestamp_col], unit="ms")
            elif med > 1e9:
                out[timestamp_col] = pd.to_datetime(out[timestamp_col], unit="s")
            else:
                out[timestamp_col] = pd.Timestamp("2000-01-01") + pd.to_timedelta(out[timestamp_col], unit="s")
        else:
            out[timestamp_col] = pd.to_datetime(out[timestamp_col])

    if timezone is not None:
        try:
            if out[timestamp_col].dt.tz is None:
                out[timestamp_col] = out[timestamp_col].dt.tz_localize(timezone)
            else:
                out[timestamp_col] = out[timestamp_col].dt.tz_convert(timezone)
        except Exception:
            pass

    out = out.sort_values(timestamp_col).set_index(timestamp_col)

    required = [price_col, volume_col, side_col, mid_col, bid_col, ask_col]
    for col in required:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=required)
    out = out[out[volume_col] > 0]
    out = out[out[side_col].isin([-1, 1])]
    out = out[out[ask_col] >= out[bid_col]]

    out["spread"] = out[ask_col] - out[bid_col]
    out["spread_ticks"] = out["spread"] / tick_size
    out["mid_return_ticks_trade_time"] = out[mid_col].diff() / tick_size
    out["trade_return_ticks"] = out[price_col].diff() / tick_size
    out["signed_qty"] = out[side_col] * out[volume_col]
    out["signed_log_qty"] = out[side_col] * np.log1p(out[volume_col])
    out["notional"] = out[price_col] * out[volume_col] * multiplier

    out["date"] = pd.to_datetime(out.index.date)
    out["hour"] = out.index.hour
    out["minute"] = out.index.minute
    out["weekday"] = out.index.day_name()

    # Convention simple : session_day = date calendaire.
    # À adapter si tu veux rattacher une session overnight à la date de settlement.
    out["session_day"] = out["date"]

    return out


## 4. Résumé quotidien

On construit une table extraday avec open/close, high/low, volume, imbalance, spreads, volatilité réalisée, range, VWAP et proxy d'illiquidité.


In [ ]:
def make_daily_summary(
    df_clean,
    price_col=PRICE_COL,
    volume_col=VOLUME_COL,
    side_col=SIDE_COL,
    mid_col=MID_COL,
    tick_size=tick_size,
    intraday_bucket=INTRADAY_BUCKET
):
    df = df_clean.copy()
    grouped = df.groupby("session_day")

    daily = pd.DataFrame({
        "first_mid": grouped[mid_col].first(),
        "last_mid": grouped[mid_col].last(),
        "high_mid": grouped[mid_col].max(),
        "low_mid": grouped[mid_col].min(),
        "first_trade_price": grouped[price_col].first(),
        "last_trade_price": grouped[price_col].last(),
        "volume": grouped[volume_col].sum(),
        "trade_count": grouped[volume_col].count(),
        "signed_volume": grouped["signed_qty"].sum(),
        "signed_log_volume": grouped["signed_log_qty"].sum(),
        "notional": grouped["notional"].sum(),
        "avg_spread_ticks": grouped["spread_ticks"].mean(),
        "median_spread_ticks": grouped["spread_ticks"].median(),
        "max_spread_ticks": grouped["spread_ticks"].max(),
        "avg_trade_size": grouped[volume_col].mean(),
        "median_trade_size": grouped[volume_col].median(),
    })

    daily["daily_return_ticks"] = (daily["last_mid"] - daily["first_mid"]) / tick_size
    daily["daily_return_bps"] = 1e4 * (daily["last_mid"] / daily["first_mid"] - 1.0)
    daily["range_ticks"] = (daily["high_mid"] - daily["low_mid"]) / tick_size
    daily["imbalance"] = daily["signed_volume"] / daily["volume"]
    daily["imbalance"] = daily["imbalance"].replace([np.inf, -np.inf], np.nan)

    daily["vwap"] = grouped.apply(
        lambda x: np.average(x[price_col], weights=x[volume_col]) if x[volume_col].sum() > 0 else np.nan
    )

    # Volatilité réalisée intraday, calculée sur mid resamplé
    mid_intraday = df[mid_col].resample(intraday_bucket).last().ffill()
    r_intraday = mid_intraday.diff() / tick_size
    rv_daily = r_intraday.pow(2).groupby(pd.to_datetime(r_intraday.index.date)).sum()
    daily["rv_ticks_sq"] = rv_daily
    daily["realized_vol_ticks"] = np.sqrt(daily["rv_ticks_sq"])

    daily["amihud_ticks_per_contract"] = daily["daily_return_ticks"].abs() / daily["volume"]

    return daily


In [ ]:
# Exemple :
# df_clean = prepare_trade_df(df)
# daily = make_daily_summary(df_clean)
# daily.head()


## 5. Graphiques extraday principaux

In [ ]:
def plot_daily_overview(daily):
    plots = [
        ("volume", "Volume quotidien"),
        ("trade_count", "Nombre de trades quotidien"),
        ("realized_vol_ticks", "Volatilité réalisée quotidienne, ticks"),
        ("avg_spread_ticks", "Spread moyen quotidien, ticks"),
        ("imbalance", "Imbalance quotidienne"),
        ("daily_return_ticks", "Rendement quotidien, ticks"),
        ("range_ticks", "Range quotidien, ticks"),
        ("amihud_ticks_per_contract", "Proxy illiquidité, |return| / volume"),
    ]

    for col, title in plots:
        if col in daily.columns:
            plt.figure(figsize=(12, 4))
            plt.plot(daily.index, daily[col], marker="o")
            plt.axhline(0, linestyle="--")
            plt.title(title)
            plt.xlabel("Date")
            plt.ylabel(col)
            plt.grid(True, alpha=0.3)
            plt.show()


In [ ]:
# plot_daily_overview(daily)


## 6. Distributions quotidiennes

But : détecter si quelques journées extrêmes dominent l'échantillon.


In [ ]:
def plot_daily_distributions(daily):
    cols = [
        "volume",
        "trade_count",
        "daily_return_ticks",
        "realized_vol_ticks",
        "range_ticks",
        "imbalance",
        "avg_spread_ticks",
        "amihud_ticks_per_contract",
    ]

    existing = [c for c in cols if c in daily.columns]

    for col in existing:
        plt.figure(figsize=(8, 4))
        daily[col].dropna().hist(bins=40)
        plt.title(f"Distribution quotidienne — {col}")
        plt.xlabel(col)
        plt.ylabel("Fréquence")
        plt.grid(True, alpha=0.3)
        plt.show()

    return daily[existing].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])


In [ ]:
# desc_daily = plot_daily_distributions(daily)
# desc_daily


## 7. Overnight, open gap et close-to-close

On distingue :

$$
R^{intraday}_d = \frac{Close_d - Open_d}{ticksize}
$$

$$
R^{overnight}_d = \frac{Open_d - Close_{d-1}}{ticksize}
$$

$$
R^{cc}_d = \frac{Close_d - Close_{d-1}}{ticksize}
$$


In [ ]:
def add_overnight_metrics(daily, tick_size=tick_size):
    out = daily.copy()
    out["prev_last_mid"] = out["last_mid"].shift(1)

    out["intraday_return_ticks"] = (out["last_mid"] - out["first_mid"]) / tick_size
    out["overnight_gap_ticks"] = (out["first_mid"] - out["prev_last_mid"]) / tick_size
    out["close_to_close_return_ticks"] = (out["last_mid"] - out["prev_last_mid"]) / tick_size

    out["overnight_gap_bps"] = 1e4 * (out["first_mid"] / out["prev_last_mid"] - 1.0)
    out["close_to_close_return_bps"] = 1e4 * (out["last_mid"] / out["prev_last_mid"] - 1.0)

    return out


In [ ]:
def plot_overnight_metrics(daily):
    cols = [
        ("overnight_gap_ticks", "Overnight gap en ticks"),
        ("intraday_return_ticks", "Intraday return en ticks"),
        ("close_to_close_return_ticks", "Close-to-close return en ticks"),
    ]

    for col, title in cols:
        if col in daily.columns:
            plt.figure(figsize=(12, 4))
            plt.plot(daily.index, daily[col], marker="o")
            plt.axhline(0, linestyle="--")
            plt.title(title)
            plt.xlabel("Date")
            plt.ylabel(col)
            plt.grid(True, alpha=0.3)
            plt.show()

    if {"overnight_gap_ticks", "intraday_return_ticks"}.issubset(daily.columns):
        tmp = daily[["overnight_gap_ticks", "intraday_return_ticks"]].dropna()
        plt.figure(figsize=(6, 6))
        plt.scatter(tmp["overnight_gap_ticks"], tmp["intraday_return_ticks"], alpha=0.7)
        plt.axhline(0, linestyle="--")
        plt.axvline(0, linestyle="--")
        plt.xlabel("Overnight gap, ticks")
        plt.ylabel("Intraday return, ticks")
        plt.title("Overnight gap vs intraday return")
        plt.grid(True, alpha=0.3)
        plt.show()


In [ ]:
# daily = add_overnight_metrics(daily)
# plot_overnight_metrics(daily)


## 8. Le flow de la veille prédit-il le gap ou le rendement du lendemain ?

On crée des variables lead/lag quotidiennes pour explorer les relations extraday.


In [ ]:
def add_lead_lag_daily_features(daily):
    out = daily.copy()

    for col in [
        "overnight_gap_ticks",
        "intraday_return_ticks",
        "close_to_close_return_ticks",
        "realized_vol_ticks",
        "volume",
        "imbalance",
        "avg_spread_ticks",
    ]:
        if col in out.columns:
            out[f"next_{col}"] = out[col].shift(-1)

    for col in [
        "volume",
        "imbalance",
        "signed_volume",
        "realized_vol_ticks",
        "daily_return_ticks",
        "range_ticks",
        "avg_spread_ticks",
    ]:
        if col in out.columns:
            out[f"lag_{col}"] = out[col].shift(1)

    return out


In [ ]:
def daily_predictive_correlations(daily):
    pairs = [
        ("imbalance", "next_overnight_gap_ticks"),
        ("imbalance", "next_intraday_return_ticks"),
        ("signed_volume", "next_overnight_gap_ticks"),
        ("signed_volume", "next_intraday_return_ticks"),
        ("volume", "next_realized_vol_ticks"),
        ("realized_vol_ticks", "next_realized_vol_ticks"),
        ("avg_spread_ticks", "next_realized_vol_ticks"),
        ("daily_return_ticks", "next_intraday_return_ticks"),
    ]

    rows = []
    for x, y in pairs:
        if x in daily.columns and y in daily.columns:
            tmp = daily[[x, y]].dropna()
            if len(tmp) > 5:
                corr = tmp[x].corr(tmp[y])
                rows.append({"x": x, "y": y, "corr": corr, "n": len(tmp)})

                plt.figure(figsize=(6, 5))
                plt.scatter(tmp[x], tmp[y], alpha=0.7)
                plt.axhline(0, linestyle="--")
                plt.axvline(0, linestyle="--")
                plt.xlabel(x)
                plt.ylabel(y)
                plt.title(f"{x} vs {y}, corr={corr:.3f}")
                plt.grid(True, alpha=0.3)
                plt.show()

    return pd.DataFrame(rows)


In [ ]:
# daily = add_lead_lag_daily_features(daily)
# corr_table = daily_predictive_correlations(daily)
# corr_table


## 9. Jours extrêmes

On flagge les journées extrêmes : volume, volatilité, range, spread, imbalance, gap.

Ces jours doivent souvent être segmentés ou traités à part avant Hasbrouck/Hawkes.


In [ ]:
def flag_extreme_days(daily, q=0.95):
    out = daily.copy()

    metrics = [
        "volume",
        "trade_count",
        "realized_vol_ticks",
        "range_ticks",
        "avg_spread_ticks",
        "amihud_ticks_per_contract",
    ]

    if "overnight_gap_ticks" in out.columns:
        out["abs_overnight_gap_ticks"] = out["overnight_gap_ticks"].abs()
        metrics.append("abs_overnight_gap_ticks")

    if "daily_return_ticks" in out.columns:
        out["abs_daily_return_ticks"] = out["daily_return_ticks"].abs()
        metrics.append("abs_daily_return_ticks")

    if "imbalance" in out.columns:
        out["abs_imbalance"] = out["imbalance"].abs()
        metrics.append("abs_imbalance")

    for m in metrics:
        if m in out.columns:
            threshold = out[m].quantile(q)
            out[f"is_extreme_{m}"] = out[m] >= threshold

    flag_cols = [c for c in out.columns if c.startswith("is_extreme_")]
    out["is_any_extreme_day"] = out[flag_cols].any(axis=1) if flag_cols else False

    return out


In [ ]:
# daily_ext = flag_extreme_days(daily, q=0.95)
# daily_ext[daily_ext['is_any_extreme_day']].tail(20)


## 10. Analyse roll

Si tu connais les dates de roll, ajoute un flag autour de ces dates.


In [ ]:
def add_manual_roll_flag(daily, roll_dates, window_days=3):
    out = daily.copy()
    out["is_roll_window"] = False

    roll_dates = pd.to_datetime(roll_dates)
    for d in roll_dates:
        start = d - pd.Timedelta(days=window_days)
        end = d + pd.Timedelta(days=window_days)
        out.loc[(out.index >= start) & (out.index <= end), "is_roll_window"] = True

    return out


def compare_flag_groups(daily, flag_col):
    if flag_col not in daily.columns:
        raise ValueError(f"{flag_col} absent du DataFrame.")

    cols = [
        "volume",
        "trade_count",
        "realized_vol_ticks",
        "range_ticks",
        "avg_spread_ticks",
        "imbalance",
        "amihud_ticks_per_contract",
    ]
    existing = [c for c in cols if c in daily.columns]
    summary = daily.groupby(flag_col)[existing].agg(["mean", "median", "std", "count"])

    for col in existing:
        plt.figure(figsize=(7, 4))
        daily.boxplot(column=col, by=flag_col)
        plt.title(f"{col} — grouped by {flag_col}")
        plt.suptitle("")
        plt.xlabel(flag_col)
        plt.ylabel(col)
        plt.grid(True, alpha=0.3)
        plt.show()

    return summary


In [ ]:
# Exemple :
# roll_dates = ['2024-03-15', '2024-06-21', '2024-09-20', '2024-12-20']
# daily_roll = add_manual_roll_flag(daily, roll_dates, window_days=3)
# roll_summary = compare_flag_groups(daily_roll, 'is_roll_window')
# roll_summary


## 11. Jours macro / news

Ajoute un calendrier d'événements macro pour comparer jours normaux vs jours d'annonce.


In [ ]:
def add_event_calendar_flag(daily, event_dates, event_name="macro"):
    out = daily.copy()
    col = f"is_{event_name}_day"
    out[col] = False

    event_dates = pd.to_datetime(event_dates)
    out.loc[out.index.normalize().isin(event_dates.normalize()), col] = True
    return out


In [ ]:
# Exemple :
# macro_dates = ['2024-01-10', '2024-02-02']
# daily_macro = add_event_calendar_flag(daily, macro_dates, event_name='macro')
# macro_summary = compare_flag_groups(daily_macro, 'is_macro_day')
# macro_summary


## 12. Stabilité mensuelle et hebdomadaire

Permet de voir si la liquidité, la volatilité ou le spread changent au fil du temps.


In [ ]:
def monthly_weekly_summary(daily):
    out = daily.copy()

    agg_map = {
        "volume": "sum",
        "trade_count": "sum",
        "realized_vol_ticks": "mean",
        "avg_spread_ticks": "mean",
        "range_ticks": "mean",
        "imbalance": "mean",
        "daily_return_ticks": "sum",
    }
    agg_map = {k: v for k, v in agg_map.items() if k in out.columns}

    monthly = out.resample("ME").agg(agg_map)
    weekly = out.resample("W").agg(agg_map)

    return monthly, weekly


def plot_periodic_summary(periodic, title_prefix="Monthly"):
    for col in periodic.columns:
        plt.figure(figsize=(12, 4))
        plt.plot(periodic.index, periodic[col], marker="o")
        plt.axhline(0, linestyle="--")
        plt.title(f"{title_prefix} — {col}")
        plt.xlabel("Date")
        plt.ylabel(col)
        plt.grid(True, alpha=0.3)
        plt.show()


In [ ]:
# monthly, weekly = monthly_weekly_summary(daily)
# plot_periodic_summary(monthly, 'Monthly')
# plot_periodic_summary(weekly, 'Weekly')


## 13. Flow quotidien vs mouvement quotidien

Diagnostic simple avant modèles dynamiques :

$$
R_d = \alpha + \beta \cdot Imbalance_d + \epsilon_d
$$

Ce n'est pas causal, mais utile pour voir si le flow agrégé de la journée est aligné avec le mouvement du mid.


In [ ]:
def plot_daily_flow_vs_return(daily):
    pairs = [
        ("imbalance", "daily_return_ticks"),
        ("signed_volume", "daily_return_ticks"),
        ("imbalance", "close_to_close_return_ticks"),
        ("signed_volume", "close_to_close_return_ticks"),
        ("volume", "range_ticks"),
        ("volume", "realized_vol_ticks"),
    ]

    rows = []
    for x, y in pairs:
        if x in daily.columns and y in daily.columns:
            tmp = daily[[x, y]].dropna()
            if len(tmp) > 5:
                corr = tmp[x].corr(tmp[y])
                rows.append({"x": x, "y": y, "corr": corr, "n": len(tmp)})

                plt.figure(figsize=(6, 5))
                plt.scatter(tmp[x], tmp[y], alpha=0.7)
                plt.axhline(0, linestyle="--")
                plt.axvline(0, linestyle="--")
                plt.title(f"{x} vs {y}, corr={corr:.3f}")
                plt.xlabel(x)
                plt.ylabel(y)
                plt.grid(True, alpha=0.3)
                plt.show()

    return pd.DataFrame(rows)


In [ ]:
# flow_return_corrs = plot_daily_flow_vs_return(daily)
# flow_return_corrs


## 14. Persistance extraday

Autocorrélation quotidienne des métriques :

$$
Corr(X_d, X_{d-k})
$$


In [ ]:
def daily_autocorrelations(daily, max_lag=10):
    cols = [
        "volume",
        "trade_count",
        "realized_vol_ticks",
        "avg_spread_ticks",
        "range_ticks",
        "imbalance",
        "daily_return_ticks",
        "close_to_close_return_ticks",
    ]

    result = {}
    for col in cols:
        if col in daily.columns:
            s = daily[col].dropna()
            acfs = [s.autocorr(lag=k) for k in range(1, max_lag + 1)]
            result[col] = acfs

            plt.figure(figsize=(8, 4))
            plt.bar(np.arange(1, max_lag + 1), acfs)
            plt.axhline(0, linestyle="--")
            plt.title(f"Autocorrélation quotidienne — {col}")
            plt.xlabel("Lag en jours")
            plt.ylabel("Autocorrélation")
            plt.grid(True, alpha=0.3)
            plt.show()

    return pd.DataFrame(result, index=np.arange(1, max_lag + 1))


In [ ]:
# acf_daily = daily_autocorrelations(daily, max_lag=10)
# acf_daily


## 15. Comparaison DAX vs SXE

Si tu as deux DataFrames, prépare-les séparément puis compare les métriques quotidiennes.


In [ ]:
def compare_two_contracts_daily(daily_a, daily_b, name_a="DAX", name_b="SXE"):
    a = daily_a.copy().add_prefix(f"{name_a}_")
    b = daily_b.copy().add_prefix(f"{name_b}_")
    combined = a.join(b, how="inner")

    metrics = [
        "volume",
        "trade_count",
        "realized_vol_ticks",
        "avg_spread_ticks",
        "imbalance",
        "daily_return_ticks",
        "range_ticks",
    ]

    for m in metrics:
        ca = f"{name_a}_{m}"
        cb = f"{name_b}_{m}"
        if ca in combined.columns and cb in combined.columns:
            corr = combined[ca].corr(combined[cb])

            plt.figure(figsize=(6, 5))
            plt.scatter(combined[ca], combined[cb], alpha=0.7)
            plt.xlabel(f"{name_a} {m}")
            plt.ylabel(f"{name_b} {m}")
            plt.title(f"{name_a} vs {name_b} — {m}, corr={corr:.3f}")
            plt.grid(True, alpha=0.3)
            plt.show()

            plt.figure(figsize=(12, 4))
            plt.plot(combined.index, combined[ca], label=name_a)
            plt.plot(combined.index, combined[cb], label=name_b)
            plt.title(f"Daily comparison — {m}")
            plt.xlabel("Date")
            plt.ylabel(m)
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()

    return combined


In [ ]:
# Exemple :
# df_dax_clean = prepare_trade_df(df_dax, tick_size=0.5, multiplier=25.0)
# daily_dax = add_overnight_metrics(make_daily_summary(df_dax_clean), tick_size=0.5)
#
# df_sxe_clean = prepare_trade_df(df_sxe, tick_size=1.0, multiplier=10.0)
# daily_sxe = add_overnight_metrics(make_daily_summary(df_sxe_clean), tick_size=1.0)
#
# combined = compare_two_contracts_daily(daily_dax, daily_sxe, name_a='DAX', name_b='SXE')
# combined.head()


## 16. Pipeline complet

Cette fonction applique le workflow principal sur un DataFrame unique.


In [ ]:
def run_extraday_descriptive_pipeline(
    df,
    tick_size=tick_size,
    multiplier=multiplier,
    roll_dates=None,
    macro_dates=None,
    plot=True
):
    df_clean = prepare_trade_df(
        df,
        tick_size=tick_size,
        multiplier=multiplier
    )

    daily = make_daily_summary(
        df_clean,
        tick_size=tick_size
    )

    daily = add_overnight_metrics(daily, tick_size=tick_size)
    daily = add_lead_lag_daily_features(daily)
    daily = flag_extreme_days(daily, q=0.95)

    if roll_dates is not None:
        daily = add_manual_roll_flag(daily, roll_dates=roll_dates, window_days=3)

    if macro_dates is not None:
        daily = add_event_calendar_flag(daily, macro_dates, event_name="macro")

    monthly, weekly = monthly_weekly_summary(daily)

    if plot:
        plot_daily_overview(daily)
        plot_overnight_metrics(daily)
        _ = plot_daily_flow_vs_return(daily)
        _ = daily_autocorrelations(daily, max_lag=10)

    return {
        "df_clean": df_clean,
        "daily": daily,
        "monthly": monthly,
        "weekly": weekly,
    }


In [ ]:
# Exemple :
# result = run_extraday_descriptive_pipeline(
#     df,
#     tick_size=0.5,
#     multiplier=25.0,
#     roll_dates=None,
#     macro_dates=None,
#     plot=True
# )
#
# daily = result['daily']
# daily.head()


## Checklist d'interprétation extraday

Avant de passer à Hasbrouck, Hawkes ou point processes, vérifie :

### Qualité
- Nombre de jours disponibles.
- Jours incomplets.
- Cohérence des horaires.
- Timestamps et timezone.
- Roll windows et jours macro.

### Liquidité
- Volume quotidien stable ou non.
- Spreads moyens.
- Nombre de trades.
- Jours de liquidité anormale.

### Risque
- Volatilité réalisée quotidienne.
- Gaps overnight.
- Range quotidien.
- Jours extrêmes.

### Flow
- Imbalance quotidienne.
- Signed volume.
- Relation flow du jour / rendement du jour.
- Relation flow de la veille / gap du lendemain.

### Comparaison SXE/DAX
- Corrélations journalières.
- Volatilité relative.
- Spread relatif.
- Volume relatif.
- Co-mouvements des jours extrêmes.

Ces analyses servent à décider comment segmenter les modèles Hasbrouck ou Hawkes : par régime, par heure, par roll, par jours macro, ou par instrument.
